## Setup

In [1]:
from dotenv import load_dotenv
from pathlib import Path
from sqlalchemy import create_engine
import utils
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.store.sqlite import SqliteStore
import json

/Users/burakkilic/msc-ai/curriculum/nd901-agentic-ai-langchain-langgraph/projects/autonomous-knowledge-agent/.venv/lib/python3.13/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
load_dotenv()

True

## Run

In [3]:
solution_dir = Path(utils.__file__).resolve().parent

cultpass_db = solution_dir / "data" / "external" / "cultpass.db"
udahub_db = solution_dir / "data" / "core" / "udahub.db"

# Check before connecting: SQLite could otherwise create an empty database.
for database_path in (cultpass_db, udahub_db):
    # api: is_file checks that an existing path points to a file.
    if not database_path.is_file():
        raise FileNotFoundError(
            f"Missing database: {database_path}. "
            "Run setup notebooks 01 and 02 first."
        )

# api: create_engine prepares database access; connections open when used.
cultpass_engine = create_engine(f"sqlite:///{cultpass_db}", echo=False)
udahub_engine = create_engine(f"sqlite:///{udahub_db}", echo=False)

print("Both database engines are ready.")

Both database engines are ready.


In [4]:
#Create persistent conversation checkpoint
memory_dir = solution_dir / "data" / "memory"

# api: mkdir creates missing parent directories and accepts an existing folder.
memory_dir.mkdir(parents=True, exist_ok=True)

checkpoint_db = memory_dir / "checkpoints.sqlite"

# api: check_same_thread=False permits LangGraph worker threads to use this connection.
checkpoint_connection = sqlite3.connect(
    str(checkpoint_db),
    check_same_thread=False,
)

# api: SqliteSaver saves graph state and conversation checkpoints in SQLite.
checkpointer = SqliteSaver(checkpoint_connection)

# api: setup creates the checkpoint tables when they do not already exist.
checkpointer.setup()

print("Conversation checkpointing is ready.")

Conversation checkpointing is ready.


In [5]:
from agentic.workflow import build_workflow, AgentModels, RetrievalSettings
from agentic.workflow import WorkflowLimits
from uuid import uuid4

from langchain_core.documents import Document
from langchain_chroma import Chroma
from data.models import udahub

limits: WorkflowLimits = {
    "max_graph_steps": 20,
    "max_model_calls": 15,
    "call_timeout_seconds": 30.0,
}

chat_model_name = "gpt-4o-mini"
embedding_model_name = "text-embedding-3-small"

if not chat_model_name or not embedding_model_name:
    raise ValueError("Set both model names before running this cell.")

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")

if not api_key:
     api_key=os.getenv("VOCAREUM_API_KEY")
     if api_key and not base_url:
          raise ValueError("Set OPENAI_BASE_URL to your Vocareum API endpoint.")

if not api_key:
    raise ValueError("Set OPENAI_API_KEY or VOCAREUM_API_KEY in your .env file.")

chat_model = ChatOpenAI(
     model_name=chat_model_name, 
     # why: temperature 0 keeps classification and structured outputs consistent.
     temperature=0,
     api_key=api_key, 
     base_url=base_url,
     timeout = limits["call_timeout_seconds"],
     max_retries = 0,
     )
embeddings = OpenAIEmbeddings(
     model=embedding_model_name,
     api_key=api_key,
     base_url=base_url,
     timeout = limits["call_timeout_seconds"],
     max_retries = 0,
)

print("Chat and embedding clients are ready.")

#Create the persistent long term memory store.

memory_db = memory_dir / "customer_memory.sqlite"

embedding_dimensions = 1536

memory_connection = sqlite3.connect(
     str(memory_db),
     check_same_thread=False,
     isolation_level=None,
)

store = SqliteStore(
     memory_connection,
     index ={
          "dims": embedding_dimensions,
          "embed": embeddings,
          "fields": ["content"],
     },
)

store.setup()

print ("Persistent long-term memory store is ready.")

# Build the cultpass article index

knowledge_account_id = "cultpass"

with utils.get_session(udahub_engine) as session:
     articles = (
          session.query(udahub.Knowledge)
          .filter(udahub.Knowledge.account_id == knowledge_account_id)
          .all()
     )

     documents= [
          Document(
               page_content=article.content,
               metadata = {
                    "article_id": article.article_id,
                    "account_id": article.account_id,
                    "title": article.title
               },
          )
          for article in articles
     ]
if not documents:
     raise ValueError("No CultPass articles were found. Complete notebook 2 first.")

collection_name = f"cultpass_articles_{uuid4().hex}"

knowledge_store = Chroma.from_documents(
     documents = documents,
     embedding = embeddings,
     ids = [document.metadata["article_id"] for document in documents],
     collection_name = collection_name,
)

print (f"Indexed {len(documents)} CultPass articles.")

# Inspect retrieval scores before selecting the gate
retrieval_test_queries = {
    "supported_login":
        "I requested a password reset but the email has not arrived.",

    "supported_reservation":
        "How can I check the status of my existing reservation?",

    "supported_availability":
        "Are there places available for the museum experience?",

    "supported_blocked_account":
        "My CultPass account is blocked. Can you restore access?",

    "supported_refund_guidance":
        "I want a refund for an unexpected CultPass charge.",

    "related_but_missing_financial_facts":
        "What exact amount will CultPass charge my bank account next month?",

    "unsupported_passport":
        "How do I renew my passport?",
}

if any(not query.strip() for query in retrieval_test_queries.values()):
     raise ValueError("Fill both retrieval test questions before running.")

retrieval_observations = []

for test_case, query in retrieval_test_queries.items():
    # api: this returns Document/score pairs matching the metadata filter.
    # Inspect every indexed article here; runtime candidate count comes later.
    candidates = knowledge_store.similarity_search_with_relevance_scores(
        query,
        k=len(documents),
        filter={"account_id": knowledge_account_id},
    )

    print(f"\n{test_case.upper()}: {query}")

    for document, score in candidates:
        retrieval_observations.append({
            "test_case": test_case,
            "query": query,
            "article_id": document.metadata["article_id"],
            "title": document.metadata["title"],
            "content": document.page_content,
            "score": float(score),
        })

        print(f"{score:.4f} | {document.metadata['title']}")



Chat and embedding clients are ready.
Persistent long-term memory store is ready.
Indexed 14 CultPass articles.


/var/folders/xb/j6kztsq52gx69y2zhmfp44qh0000gn/T/ipykernel_4611/3537265159.py:144: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='af239b86-13ed-45dd-a8d4-ed9df1c337fa', metadata={'article_id': 'af239b86-13ed-45dd-a8d4-ed9df1c337fa', 'account_id': 'cultpass', 'title': 'Password Reset Email Has Not Arrived'}, page_content="Ask the customer to use 'Forgot Password' on the CultPass login screen and check that they entered their registered email. Suggest checking the spam folder. If the reset message has not arrived, retry after 10 minutes, as described in the login guidance. Persistent reset-email or login problems require human support. Do not claim that support sent an email or confirmed its delivery."), 0.29807593248114017), (Document(id='08d3bd4e-bf0b-480a-a935-5a54c99f23f1', metadata={'article_id': '08d3bd4e-bf0b-480a-a935-5a54c99f23f1', 'account_id': 'cultpass', 'title': 'How to Handle Login Issues?'}, page_content='Most login issues are resolved with passw


SUPPORTED_LOGIN: I requested a password reset but the email has not arrived.
0.2981 | Password Reset Email Has Not Arrived
0.1949 | How to Handle Login Issues?
-0.0512 | Refund Requests and Unexplained Charges
-0.1183 | Cancellation and Pause Requests: What Support Can Confirm
-0.1403 | Support for a Blocked CultPass Account
-0.1411 | Finding and Checking an Existing Reservation
-0.1476 | How to Reserve a Spot for an Event
-0.1981 | Checking the Current Subscription Status
-0.2034 | Checking Available Places Before Booking
-0.2100 | Checking an Experience's Date and Location
-0.2393 | How to Cancel or Pause a Subscription
-0.2480 | Monthly Allowance and Remaining Experiences
-0.2487 | Premium Experiences and Additional Charges
-0.2673 | What's Included in a CultPass Subscription


/var/folders/xb/j6kztsq52gx69y2zhmfp44qh0000gn/T/ipykernel_4611/3537265159.py:144: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='6c16bbe2-274a-47be-8896-6d28560342b8', metadata={'account_id': 'cultpass', 'article_id': '6c16bbe2-274a-47be-8896-6d28560342b8', 'title': 'Finding and Checking an Existing Reservation'}, page_content='Retrieve reservations only for the customer linked to the support case. Use a reservation ID when available, or ask for event title, date, or location to identify the intended booking. If several records match a request for one booking, ask which booking the customer means. Report the verified reservation status and associated event details. Do not claim that a reservation was created, changed, or cancelled by the support assistant.'), 0.2558773276402747), (Document(id='e96dc17f-a79f-415f-ab9f-366890df3ec8', metadata={'title': 'Checking Available Places Before Booking', 'article_id': 'e96dc17f-a79f-415f-ab9f-366890df3ec8', 'account_id


SUPPORTED_RESERVATION: How can I check the status of my existing reservation?
0.2559 | Finding and Checking an Existing Reservation
0.1732 | Checking Available Places Before Booking
0.1257 | Support for a Blocked CultPass Account
0.1069 | Checking the Current Subscription Status
0.0717 | How to Reserve a Spot for an Event
0.0297 | Refund Requests and Unexplained Charges
0.0235 | Cancellation and Pause Requests: What Support Can Confirm
0.0186 | Monthly Allowance and Remaining Experiences
-0.0122 | Premium Experiences and Additional Charges
-0.0183 | Checking an Experience's Date and Location
-0.0462 | How to Cancel or Pause a Subscription
-0.0660 | How to Handle Login Issues?
-0.1210 | Password Reset Email Has Not Arrived
-0.1613 | What's Included in a CultPass Subscription


...

/var/folders/xb/j6kztsq52gx69y2zhmfp44qh0000gn/T/ipykernel_4611/3537265159.py:144: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='d879a933-cc51-4ab6-8d20-5b61dde26360', metadata={'article_id': 'd879a933-cc51-4ab6-8d20-5b61dde26360', 'account_id': 'cultpass', 'title': 'Refund Requests and Unexplained Charges'}, page_content="Refund offers require support-lead approval, as stated in the cancellation guidance. Preserve the customer's refund request or charge concern for human review. The available CultPass records do not include a payment ledger, so subscription or reservation data cannot verify a debit amount, payment failure, or completed refund. Do not promise eligibility, a refund amount, processing time, or that money has been returned."), 0.48198454889539644), (Document(id='4c606fd3-d6ad-4438-bce8-62cf32387a6d', metadata={'title': 'Cancellation and Pause Requests: What Support Can Confirm', 'account_id': 'cultpass', 'article_id': '4c606fd3-d6ad-4438-bce8-6


SUPPORTED_REFUND_GUIDANCE: I want a refund for an unexpected CultPass charge.
0.4820 | Refund Requests and Unexplained Charges
0.3116 | Cancellation and Pause Requests: What Support Can Confirm
0.2378 | How to Cancel or Pause a Subscription
0.2277 | Password Reset Email Has Not Arrived
0.1770 | What's Included in a CultPass Subscription
0.0773 | How to Reserve a Spot for an Event
0.0587 | Checking Available Places Before Booking
0.0156 | Premium Experiences and Additional Charges
-0.0186 | Checking the Current Subscription Status
-0.0369 | Monthly Allowance and Remaining Experiences
-0.0807 | Support for a Blocked CultPass Account
-0.1708 | How to Handle Login Issues?
-0.1774 | Finding and Checking an Existing Reservation
-0.1996 | Checking an Experience's Date and Location


/var/folders/xb/j6kztsq52gx69y2zhmfp44qh0000gn/T/ipykernel_4611/3537265159.py:144: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='0d139ec7-2cad-450f-87e2-68fa7e119ae9', metadata={'account_id': 'cultpass', 'article_id': '0d139ec7-2cad-450f-87e2-68fa7e119ae9', 'title': "What's Included in a CultPass Subscription"}, page_content='Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass subscription includes 4 curated experiences each month. You can choose from museums, concerts, film events and more. Premium events may have an extra cost, which is shown during reservation."'), 0.3850950976087337), (Document(id='4c606fd3-d6ad-4438-bce8-62cf32387a6d', metadata={'title': 'Cancellation and Pause Requests: What Support Can Confirm', 'account_id


RELATED_BUT_MISSING_FINANCIAL_FACTS: What exact amount will CultPass charge my bank account next month?
0.3851 | What's Included in a CultPass Subscription
0.3536 | Cancellation and Pause Requests: What Support Can Confirm
0.2806 | How to Cancel or Pause a Subscription
0.2802 | Refund Requests and Unexplained Charges
0.2640 | Password Reset Email Has Not Arrived
0.1134 | How to Reserve a Spot for an Event
0.0829 | Premium Experiences and Additional Charges
0.0542 | Monthly Allowance and Remaining Experiences
0.0486 | Checking Available Places Before Booking
-0.0898 | Checking the Current Subscription Status
-0.1898 | Support for a Blocked CultPass Account
-0.2316 | How to Handle Login Issues?
-0.2503 | Checking an Experience's Date and Location
-0.2728 | Finding and Checking an Existing Reservation

UNSUPPORTED_PASSPORT: How do I renew my passport?
-0.0627 | Cancellation and Pause Requests: What Support Can Confirm
-0.0686 | Password Reset Email Has Not Arrived
-0.0829 | How to Reserv

/var/folders/xb/j6kztsq52gx69y2zhmfp44qh0000gn/T/ipykernel_4611/3537265159.py:144: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='4c606fd3-d6ad-4438-bce8-62cf32387a6d', metadata={'account_id': 'cultpass', 'article_id': '4c606fd3-d6ad-4438-bce8-62cf32387a6d', 'title': 'Cancellation and Pause Requests: What Support Can Confirm'}, page_content="Customers can manage cancellation or pause through the CultPass app under 'My Account' and 'Manage Plan'. The supplied guidance states that cancellation takes effect at the end of the billing cycle and that pausing preserves user data. The support assistant can explain this guidance and inspect the stored subscription, but cannot execute either change. Requests needing action or verification beyond these records require human review. Do not claim an effective date, completed cancellation, or automatic change without supporting evidence."), -0.06271053166617913), (Document(id='af239b86-13ed-45dd-a8d4-ed9df1c337fa', metadat

In [6]:
# Retrieval settings recorded once from the score probe above (2026-10-12).
# why: Fixed values keep every automatic run identical; the AI recommendation
# varied between runs on identical scores. Recalibrate with the probe if the
# articles, embedding model or vector-store distance setting change.
calibration_conditions = {
    "embedding_model": "text-embedding-3-small",
    "article_count": 14,
}
retrieval_threshold = 0.15
retrieval_candidate_count = 3

current_conditions = {
    "embedding_model": embedding_model_name,
    "article_count": len(documents),
}
if current_conditions != calibration_conditions:
    print(
        "WARNING: retrieval settings were calibrated for",
        calibration_conditions,
        "but the current setup is",
        current_conditions,
        "- rerun the score probe and record new values.",
    )

retrieval_settings: RetrievalSettings = {
    "threshold": retrieval_threshold,
    "candidate_count": retrieval_candidate_count,
    "embedding_model": embedding_model_name,
    "score_convention": "higher_is_more_relevant",
}

print("Retrieval settings are ready:", retrieval_settings)


Retrieval settings are ready: {'threshold': 0.15, 'candidate_count': 3, 'embedding_model': 'text-embedding-3-small', 'score_convention': 'higher_is_more_relevant'}


In [7]:
from math import isfinite

calibration_rows = []

for test_case in retrieval_test_queries:
    rows = [
        row for row in retrieval_observations
        if row["test_case"] == test_case
    ][:retrieval_settings["candidate_count"]]

    for row in rows:
        score = row["score"]

        # api: isfinite rejects infinity and NaN.
        valid_score = isfinite(score) and 0.0 <= score <= 1.0

        calibration_rows.append({
            **row,
            "threshold": retrieval_settings["threshold"],
            "score_valid": valid_score,
            "gate_pass": (
                valid_score
                and score >= retrieval_settings["threshold"]
            ),
        })

calibration_report = {
    "settings": dict(retrieval_settings),
    "calibration_conditions": calibration_conditions,
    "results": calibration_rows,
}

print(json.dumps(calibration_report, indent=2, ensure_ascii=False))

{
  "settings": {
    "threshold": 0.15,
    "candidate_count": 3,
    "embedding_model": "text-embedding-3-small",
    "score_convention": "higher_is_more_relevant"
  },
  "calibration_conditions": {
    "embedding_model": "text-embedding-3-small",
    "article_count": 14
  },
  "results": [
    {
      "test_case": "supported_login",
      "query": "I requested a password reset but the email has not arrived.",
      "article_id": "af239b86-13ed-45dd-a8d4-ed9df1c337fa",
      "title": "Password Reset Email Has Not Arrived",
      "content": "Ask the customer to use 'Forgot Password' on the CultPass login screen and check that they entered their registered email. Suggest checking the spam folder. If the reset message has not arrived, retry after 10 minutes, as described in the login guidance. Persistent reset-email or login problems require human support. Do not claim that support sent an email or confirmed its delivery.",
      "score": 0.29807593248114017,
      "threshold": 0.15,
  

In [8]:
# Bind the models and compile the workflow

models: AgentModels = {
    "supervisor": chat_model,
    "classifier": chat_model,
    "knowledge": chat_model,
    "operations": chat_model,
    "response": chat_model,
    "escalation": chat_model
}

orchestrator = build_workflow(
    cultpass_engine = cultpass_engine,
    udahub_engine = udahub_engine,
    models=models,
    knowledge_store = knowledge_store,
    retrieval_settings=retrieval_settings,
    limits=limits,
    checkpointer=checkpointer,
    store=store,
)

print("Workflow compiled successfully.")

#graph structure and diagram text.
print(orchestrator.get_graph().draw_mermaid())
demo_runs = []

Workflow compiled successfully.
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	intake(intake)
	supervisor(supervisor)
	classifier(classifier)
	knowledge(knowledge)
	operations(operations)
	response(response)
	escalation(escalation)
	persist(persist)
	__end__([<p>__end__</p>]):::last
	__start__ --> intake;
	classifier --> supervisor;
	escalation --> supervisor;
	intake --> supervisor;
	knowledge --> supervisor;
	operations --> supervisor;
	response --> supervisor;
	supervisor -.-> classifier;
	supervisor -.-> escalation;
	supervisor -.-> knowledge;
	supervisor -.-> operations;
	supervisor -.-> persist;
	supervisor -.-> response;
	persist --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [9]:
# Process pending database messages and display each saved result.
demo_runs = utils.batch_interface(
    orchestrator,
    udahub_engine,
    account_id=knowledge_account_id,
)

if not demo_runs:
    print("No pending customer messages on open tickets.")


/Users/burakkilic/msc-ai/curriculum/nd901-agentic-ai-langchain-langgraph/projects/autonomous-knowledge-agent/solution/agentic/tools/knowledge_tools.py:160: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='af239b86-13ed-45dd-a8d4-ed9df1c337fa', metadata={'account_id': 'cultpass', 'article_id': 'af239b86-13ed-45dd-a8d4-ed9df1c337fa', 'title': 'Password Reset Email Has Not Arrived'}, page_content="Ask the customer to use 'Forgot Password' on the CultPass login screen and check that they entered their registered email. Suggest checking the spam folder. If the reset message has not arrived, retry after 10 minutes, as described in the login guidance. Persistent reset-email or login problems require human support. Do not claim that support sent an email or confirmed its delivery."), 0.25613644646491973), (Document(id='08d3bd4e-bf0b-480a-a935-5a54c99f23f1', metadata={'article_id': '08d3bd4e-bf0b-480a-a935-5a54c99f23f1', 'title': 'How to Handle Login Issues?', 'account_


Ticket: 24df20ba-7cd9-443e-9de3-cac4e2478375
Outcome: resolved
Subject: Instructions to Reset Your Password

Hello,

To reset your password, please follow these steps:

1. Go to the CultPass login screen.
2. Click on 'Forgot Password'.
3. Enter your registered email address.
4. Check your email inbox for the reset email. If you don't see it, please check your spam folder.
5. If the reset email does not arrive after 10 minutes, try the process again.

If you continue to experience issues, please reach out for further assistance.

Best regards,


/Users/burakkilic/msc-ai/curriculum/nd901-agentic-ai-langchain-langgraph/projects/autonomous-knowledge-agent/solution/agentic/tools/knowledge_tools.py:160: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='af239b86-13ed-45dd-a8d4-ed9df1c337fa', metadata={'title': 'Password Reset Email Has Not Arrived', 'account_id': 'cultpass', 'article_id': 'af239b86-13ed-45dd-a8d4-ed9df1c337fa'}, page_content="Ask the customer to use 'Forgot Password' on the CultPass login screen and check that they entered their registered email. Suggest checking the spam folder. If the reset message has not arrived, retry after 10 minutes, as described in the login guidance. Persistent reset-email or login problems require human support. Do not claim that support sent an email or confirmed its delivery."), 0.29704906666402076), (Document(id='08d3bd4e-bf0b-480a-a935-5a54c99f23f1', metadata={'article_id': '08d3bd4e-bf0b-480a-a935-5a54c99f23f1', 'account_id': 'cultpass', 'title': 'How to Handle


Ticket: 0cf56053-614f-4b59-8fe5-90f6a3f8b7a5
Outcome: resolved
Subject: Password Reset Steps

Hello,

To reset your password, please follow these steps:

1. Go to the CultPass login screen.
2. Click on 'Forgot Password'.
3. Enter your registered email address.
4. Check your email inbox for the reset email. If you don't see it, please check your spam folder.
5. If the reset email does not arrive after 10 minutes, try the process again.

If you continue to experience issues, please reach out for further assistance.

Best regards,


/Users/burakkilic/msc-ai/curriculum/nd901-agentic-ai-langchain-langgraph/projects/autonomous-knowledge-agent/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=OperationsLookupPlan(issu... experience_mode=None)]), input_type=OperationsLookupPlan])
  return self.__pydantic_serializer__.to_python(
/Users/burakkilic/msc-ai/curriculum/nd901-agentic-ai-langchain-langgraph/projects/autonomous-knowledge-agent/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ResponseGeneration(draft=...tions=[]), blocker=None), input_type=ResponseGeneration])
  return self.__pydantic_serializer__.to_python(
/Users/burakkilic/msc-ai/curriculum/nd901-agenti

...

/Users/burakkilic/msc-ai/curriculum/nd901-agentic-ai-langchain-langgraph/projects/autonomous-knowledge-agent/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=OperationsLookupPlan(issu... experience_mode=None)]), input_type=OperationsLookupPlan])
  return self.__pydantic_serializer__.to_python(
/Users/burakkilic/msc-ai/curriculum/nd901-agentic-ai-langchain-langgraph/projects/autonomous-knowledge-agent/solution/agentic/tools/knowledge_tools.py:160: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='af239b86-13ed-45dd-a8d4-ed9df1c337fa', metadata={'account_id': 'cultpass', 'article_id': 'af239b86-13ed-45dd-a8d4-ed9df1c337fa', 'title': 'Password Reset Email Has Not Arrived'}, page_content="Ask the customer to use 'Forgot Password' on the CultPass login screen and check that they entered t


Ticket: cb43e6c4-7976-46bb-9b9a-a30338869d3b
Outcome: escalated
Subject: Your support request

Hello Bob,

Your request to update the email address registered on your CultPass account has been received. It has been passed to our human support team for further assistance. Thank you for your patience!

Best regards,


/Users/burakkilic/msc-ai/curriculum/nd901-agentic-ai-langchain-langgraph/projects/autonomous-knowledge-agent/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HandoffSummary(highlight_...sted_investigation=None), input_type=HandoffSummary])
  return self.__pydantic_serializer__.to_python(
/Users/burakkilic/msc-ai/curriculum/nd901-agentic-ai-langchain-langgraph/projects/autonomous-knowledge-agent/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ResponseGeneration(draft=...tions=[]), blocker=None), input_type=ResponseGeneration])
  return self.__pydantic_serializer__.to_python(



Ticket: 158a9d0b-0637-44d0-9b98-a25ecef9121d
Outcome: escalated
Your request has been received and has been passed to human support for further assistance. Thank you for your patience as we work to resolve this issue.


/Users/burakkilic/msc-ai/curriculum/nd901-agentic-ai-langchain-langgraph/projects/autonomous-knowledge-agent/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ResponseReview(draft_id='...r further assistance.']), input_type=ResponseReview])
  return self.__pydantic_serializer__.to_python(


In [10]:
# Inspect every attempted ticket after automatic processing, before cleanup.
from agentic.tools.memory_tools import search_customer_memory

demo_inspections = []
seen_tickets = set()
for run in demo_runs:
    inspected_ticket_id = run["ticket_id"]
    if inspected_ticket_id in seen_tickets:
        continue
    seen_tickets.add(inspected_ticket_id)
    inspection_config = {"configurable": {"thread_id": inspected_ticket_id}}
    try:
        # api: get_state_history yields checkpoints scoped to this thread.
        history = list(orchestrator.get_state_history(config=inspection_config))
        # api: get_state returns the latest saved values and unfinished nodes.
        snapshot = orchestrator.get_state(inspection_config)
        latest = snapshot.values
        stored_memory = (
            search_customer_memory(latest["account_id"], latest["customer_id"], store=store)
            if latest.get("account_id") and latest.get("customer_id") else []
        )
        demo_inspections.append({
            "ticket_id": inspected_ticket_id,
            "checkpoint_count": len(history),
            "pending_nodes": list(snapshot.next),
            "outcome": latest.get("outcome"),
            "messages": [{"id": msg.id, "role": msg.type, "content": msg.content}
                         for msg in latest.get("messages", [])],
            "tool_usage": latest.get("tool_results", []),
            "trace": latest.get("trace", []),
            "stored_memory": stored_memory,
            "turns": [item for item in demo_runs if item["ticket_id"] == inspected_ticket_id],
        })
    except Exception as exc:
        demo_inspections.append({"ticket_id": inspected_ticket_id,
                                 "inspection_error": type(exc).__name__})

# These are operator inspection records, including approved conversation history.
print(json.dumps(demo_inspections, indent=2, ensure_ascii=False))


[
  {
    "ticket_id": "24df20ba-7cd9-443e-9de3-cac4e2478375",
    "checkpoint_count": 11,
    "pending_nodes": [],
    "outcome": "resolved",
    "messages": [
      {
        "id": "eede8113-b435-4677-843e-6e56e23a82c4",
        "role": "human",
        "content": "How do I reset my password? Please keep replies concise."
      },
      {
        "id": "eede8113-b435-4677-843e-6e56e23a82c4:reply",
        "role": "ai",
        "content": "Subject: Instructions to Reset Your Password\n\nHello,\n\nTo reset your password, please follow these steps:\n\n1. Go to the CultPass login screen.\n2. Click on 'Forgot Password'.\n3. Enter your registered email address.\n4. Check your email inbox for the reset email. If you don't see it, please check your spam folder.\n5. If the reset email does not arrive after 10 minutes, try the process again.\n\nIf you continue to experience issues, please reach out for further assistance.\n\nBest regards,"
      }
    ],
    "tool_usage": [],
    "trace": [
  

In [11]:
# Close database resources after processing and inspection.


checkpoint_connection.close()
memory_connection.close()


cultpass_engine.dispose()
udahub_engine.dispose()

print("Database resources closed.")

Database resources closed.
